In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.retrieval.hybrid import HybridRetriever

from app.conversation.recommender import LLMRecommender
from app.agent.service import AgentService

In [2]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
print(df.shape)
df.head()

(377, 21)


,entity_id,name,link,scraped_at,job_levels,job_levels_raw,languages,languages_raw,duration,duration_raw,...,remote,adaptive,description,keys,search_document,duration_minutes,is_report,assessment_type,job_level,language
0,4302,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,2026-05-08T10:40:21.464836+00:00,"['Director', 'Entry-Level', 'Executive', 'Gene...","Director, Entry-Level, Executive, General Popu...",[],NaN,NaN,NaN,...,yes,no,This report is designed to be given to individ...,"['Ability & Aptitude', 'Assessment Exercises',...",Name: Global Skills Development Report\nDescri...,NaN,True,"Ability & Aptitude, Assessment Exercises, Biod...","Director, Entry-Level, Executive, General Popu...",NaN
1,3827,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,2026-05-08T10:39:46.810448+00:00,"['Professional Individual Contributor', 'Mid-P...","Professional Individual Contributor, Mid-Profe...",['English (USA)'],"English (USA),",30 minutes,Approximate Completion Time in minutes = 30,...,yes,yes,The.NET Framework 4.5 test measures knowledge ...,['Knowledge & Skills'],Name: .NET Framework 4.5\nDescription: The.NET...,30.0,False,Knowledge & Skills,"Professional Individual Contributor, Mid-Profe...",English (USA)
2,4094,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:39:53.276083+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",17 minutes,Approximate Completion Time in minutes = 17,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET MVC (New)\nDescription: Multi-choic...,17.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)
3,4099,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:41:53.047159+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",5 minutes,Approximate Completion Time in minutes = 5,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET MVVM (New)\nDescription: Multi-choi...,5.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)
4,4018,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:41:58.935422+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",11 minutes,Approximate Completion Time in minutes = 11,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET WCF (New)\nDescription: Multi-choic...,11.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)


In [3]:
embeddings = np.load(
    "../data/embeddings/catalog_embeddings.npy"
)

print(embeddings.shape)

(377, 384)


In [4]:
bm25 = BM25Retriever(df)

faiss = FaissRetriever(
    df,
    embeddings
)

hybrid = HybridRetriever(
    bm25,
    faiss
)

In [5]:
embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

llm = LLMRecommender()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
agent = AgentService(
    hybrid,
    embedder,
    llm
)

In [7]:
conversation = [
    {"role": "user", "content": "Hello"}
]

response = agent.chat(conversation)
print(response)

{'reply': 'Hello! How can I help you choose SHL assessments today?', 'recommendations': []}


In [8]:
conversation = [
    {"role": "user", "content": "Need assessment"}
]

response = agent.chat(conversation)
print(response)

{'reply': 'Which role are you hiring for?', 'recommendations': []}


In [9]:
conversation = [
    {"role": "user", "content": "Graduate Java Engineer"}
]

response = agent.chat(conversation)

print(response["reply"])
print(len(response["recommendations"]))

Here are the explanations for each recommended SHL assessment:

**Graduate Scenarios Narrative Report**

* Matches the hiring need: This assessment helps evaluate a candidate's ability to communicate their thought process and reasoning behind their decisions in a narrative format, which is essential for a Graduate Java Engineer role.
* Skills evaluated:
	+ Critical thinking
	+ Problem-solving
	+ Communication skills

**Graduate Scenarios**

* Matches the hiring need: This assessment assesses a candidate's ability to make sound judgments and decisions in a variety of scenarios, demonstrating their professional competence and behavior in a practical setting.
* Skills evaluated:
	+ Decision-making
	+ Time management
	+ Adaptability
2


In [10]:
print(agent.state)

ConversationState(role=Graduate Java Engineer, seniority=Graduate, skills=['Java'], assessment_types=[], duration=None, language=None)


In [11]:
conversation = [
    {"role": "user", "content": "Also include personality"}
]

response = agent.chat(conversation)

print(response["reply"])
print()
print(response["recommendations"])
print()
print(agent.state)

Here are my explanations for each recommended SHL assessment:

**1. Graduate Scenarios Narrative Report**

This assessment matches the hiring need because it evaluates a candidate's ability to articulate their thoughts, experiences, and decisions in a clear and concise manner.

Skills evaluated: Communication, Problem-solving, Critical thinking, Decision-making, Emotional intelligence.

The narrative report format allows candidates to showcase their written communication skills, provide context for their responses, and demonstrate how they apply problem-solving and critical thinking skills to real-world scenarios.

**2. Graduate Scenarios**

This assessment matches the hiring need because it assesses a candidate's ability to analyze situations, make sound judgments, and take effective actions in a business setting.

Skills evaluated: Situational awareness, Decision-making, Problem-solving, Time management, Leadership (for higher-level job roles).

The untimed format ensures that candid

In [12]:
print("Stored role:", agent.state.role)
print("Stored skills:", agent.state.skills)
print("Stored assessment types:", agent.state.assessment_types)
print("Previous results:", len(agent.state.previous_results))

Stored role: Graduate Java Engineer
Stored skills: ['Java']
Stored assessment types: ['Personality']
Previous results: 2


In [13]:
conversation = [
    {"role": "user", "content": "Remove OPQ"}
]

response = agent.chat(conversation)

print(response["reply"])
print(response["recommendations"])

Here are my explanations for each recommended SHL assessment:

**1. Graduate Scenarios Narrative Report**

* Matches the hiring need: Evaluates a candidate's ability to provide detailed and coherent responses to hypothetical scenarios, demonstrating their problem-solving skills and fit for the role.
* Skills evaluated: Critical thinking, communication, and analysis.

**2. Graduate Scenarios**

* Matches the hiring need: Assesses a candidate's situational judgment, decision-making, and behavioral skills, ensuring they can navigate complex situations in a business environment.
* Skills evaluated: Situational awareness, problem-solving, decision-making, and behavioral adaptability.
[{'index': 128, 'score': 12.475094788120424, 'name': 'Graduate Scenarios Narrative Report', 'url': 'https://www.shl.com/products/product-catalog/view/graduate-scenarios-narrative-report/', 'assessment_type': 'Biodata & Situational Judgment', 'job_level': 'Mid-Professional, Professional Individual Contributor, M

In [14]:
conversation = [
    {
        "role": "user",
        "content": "Difference between OPQ and Verify G"
    }
]

response = agent.chat(conversation)

print(response["reply"])

As an SHL consultant, I'd be happy to help compare the OPQ and Verify G+ tools.

**Purpose:**

* OPQ (Occupational Personality Questionnaire): Measures personality traits that are relevant to job performance and organizational success. It's a broad assessment tool that provides insights into an individual's behavioral tendencies.
* Verify Interactive Ability Report: Evaluates an individual's behavioral competencies, such as interpersonal skills, adaptability, and problem-solving abilities.

**What each measures:**

* OPQ: Personality traits (such as Extraversion, Conscientiousness, Openness to experience)
* Verify Interactive Ability Report: Behavioral competencies (such as teamwork, adaptability, and decision-making)

**When to choose each:**

* Use OPQ when:
	+ You need a comprehensive personality assessment.
	+ You're evaluating candidates for leadership or management roles.
	+ You want to understand an individual's core values and work style.
* Use Verify Interactive Ability Report

In [15]:
conversation = [
    {
        "role": "user",
        "content": "Thanks"
    }
]

response = agent.chat(conversation)

print(response)

{'reply': "You're welcome. Let me know if you need another assessment recommendation.", 'recommendations': []}


In [16]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer under 20 minutes"
    }
]

response = agent.chat(conversation)

print(response["reply"])
print()

for r in response["recommendations"]:
    print(r["name"], "-", r["duration"])

print()
print(agent.state)

Here are my recommendations for SHL assessments to support your hiring needs as a recruiter:

**1. Graduate Scenarios Narrative Report**

* Matches hiring need: This assessment helps evaluate a candidate's ability to provide clear and concise narrative responses, which is essential in a professional setting.
* Skills evaluated: Communication skills, problem-solving, critical thinking, and professionalism.

This assessment assesses the candidate's writing skills, experience, and maturity level, providing valuable insights into their fit for the role.

**2. Graduate Scenarios**

* Matches hiring need: This assessment evaluates a candidate's ability to make sound judgments in realistic work scenarios, which is crucial for a Java Engineer graduate.
* Skills evaluated: Problem-solving, critical thinking, decision-making, and technical skills (Java programming).

This untimed assessment assesses the candidate's analytical skills, ability to apply theoretical knowledge to real-world problems,

In [17]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer cognitive"
    }
]

response = agent.chat(conversation)

print(response["reply"])
print()

for r in response["recommendations"]:
    print(r["assessment_type"])

print()
print(agent.state)

Here are my explanations for each recommended SHL assessment:

**Graduate Scenarios Narrative Report**

This assessment matches the hiring need because it assesses a candidate's ability to think critically and make decisions in complex, dynamic situations. It evaluates their communication skills, as they will be required to write a detailed report explaining their actions.

Skills evaluated: Critical thinking, problem-solving, decision-making, communication, analysis, and reporting.

**Graduate Scenarios**

This assessment matches the hiring need because it assesses a candidate's ability to think critically and make decisions in complex situations. It evaluates their problem-solving skills, as they will be required to choose from multiple options and justify their choice.

Skills evaluated: Problem-solving, decision-making, critical thinking, analysis, and judgment.

Biodata & Situational Judgment
Biodata & Situational Judgment

ConversationState(role=Graduate Java Engineer, seniority=

In [18]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer personality under 20 minutes"
    }
]

response = agent.chat(conversation)

print(response["reply"])
print()

for r in response["recommendations"]:
    print(r["name"], r["duration"], r["assessment_type"])

print()
print(agent.state)

Here's my response as an SHL assessment recommendation assistant:

**Assessment 1: Graduate Scenarios Narrative Report**

This assessment matches the hiring need for a Graduate Java Engineer role, as it assesses the candidate's ability to think critically and solve problems in a narrative format. This skill is essential for a software engineer, as they will be required to diagnose and fix issues in code.

Skills evaluated:

* Critical thinking
* Problem-solving
* Communication

**Assessment 2: Graduate Scenarios**

This assessment also matches the hiring need for a Graduate Java Engineer role, as it evaluates the candidate's ability to make decisions and take actions in a given scenario. This skill is crucial for a software engineer, as they will be required to work independently and make technical decisions.

Skills evaluated:

* Decision-making
* Problem-solving
* Time management

Both assessments are suitable for a Graduate Java Engineer role, as they assess the candidate's technica

In [19]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer"
    }
]

response = agent.chat(conversation)

print(len(response["recommendations"]))

for r in response["recommendations"]:
    print(r["name"])

2
Graduate Scenarios Narrative Report
Graduate Scenarios


In [20]:
conversation = [
    {
        "role": "user",
        "content": "Remove Narrative"
    }
]

response = agent.chat(conversation)

print(len(response["recommendations"]))

for r in response["recommendations"]:
    print(r["name"])

1
Graduate Scenarios


In [21]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer"
    }
]

agent.chat(conversation)

{'reply': "Here's my explanation for each recommended SHL assessment:\n\n**1. Graduate Scenarios Narrative Report**\n\n* Matches the hiring need: This assessment evaluates a candidate's ability to think critically and make informed decisions in complex, dynamic situations, which is essential for a Java Engineer role.\n* Skills evaluated: Critical thinking, problem-solving, decision-making, communication, and professional judgment.\n\nThe narrative report assesses how a candidate would handle hypothetical scenarios, providing insight into their thought process, approach, and potential solutions. This helps the recruiter understand how well the candidate can navigate the technical and non-technical aspects of the role.\n\n**2. Graduate Scenarios**\n\n* Matches the hiring need: Similar to the previous assessment, this one evaluates a candidate's ability to think critically and make informed decisions in complex situations.\n* Skills evaluated: Critical thinking, problem-solving, decision-

In [22]:
conversation = [
    {
        "role": "user",
        "content": "Also include cognitive"
    }
]

response = agent.chat(conversation)

print(response["reply"])
print()

for r in response["recommendations"]:
    print(r["name"], "-", r["assessment_type"])

Here's my explanation of the recommended SHL assessments for a Graduate Java Engineer position:

**1. Graduate Scenarios Narrative Report**

This assessment matches the hiring need as it evaluates the candidate's ability to communicate their thought process and decision-making skills in a narrative format.

The skills evaluated include: Critical thinking, problem-solving, Communication, and Decision-making.

**2. Graduate Scenarios**

This assessment also matches the hiring need, focusing on the candidate's situational judgment and behavioral responses to real-world scenarios.

The skills evaluated include: Situational awareness, Problem-solving, Adaptability, and Leadership (in case of a team-based scenario).

Graduate Scenarios Narrative Report - Biodata & Situational Judgment
Graduate Scenarios - Biodata & Situational Judgment


In [23]:
print("Role:", agent.state.role)
print("Seniority:", agent.state.seniority)
print("Skills:", agent.state.skills)
print("Assessment Types:", agent.state.assessment_types)
print("Duration:", agent.state.max_duration)
print("Previous Results:", len(agent.state.previous_results))

Role: Graduate Java Engineer
Seniority: Graduate
Skills: ['Java']
Assessment Types: ['Cognitive']
Duration: None
Previous Results: 2


In [24]:
conversation = [
    {
        "role": "user",
        "content": "Graduate Java Engineer personality under 20 minutes"
    }
]

response = agent.chat(conversation)

print(agent.state)

ConversationState(role=Graduate Java Engineer, seniority=Graduate, skills=['Java'], assessment_types=['Cognitive', 'Personality'], duration=20, language=None)


In [25]:
conversation = [
    {"role": "user", "content": "Graduate Java Engineer"}
]

agent.chat(conversation)
print(agent.state)

ConversationState(role=Graduate Java Engineer, seniority=Graduate, skills=['Java'], assessment_types=[], duration=None, language=None)


In [26]:
conversation = [
    {"role": "user", "content": "Also include personality"}
]

agent.chat(conversation)
print(agent.state)

ConversationState(role=Graduate Java Engineer, seniority=Graduate, skills=['Java'], assessment_types=['Personality'], duration=None, language=None)


In [27]:
conversation = [
    {"role": "user", "content": "Sales Manager"}
]

agent.chat(conversation)
print(agent.state)

ConversationState(role=Sales Manager, seniority=Manager, skills=['Sales'], assessment_types=[], duration=None, language=None)


In [28]:
print(len(agent.state.previous_results))

6


In [29]:
conversation = [
    {"role":"user","content":"Graduate Java Engineer"}
]

response = agent.chat(conversation)

print(len(response["recommendations"]))

2


In [30]:
conversation = [
    {"role":"user","content":"Also include personality"}
]

response = agent.chat(conversation)

print(len(response["recommendations"]))

2


In [31]:
conversation = [
    {"role":"user","content":"Remove Narrative"}
]

response = agent.chat(conversation)

print(len(response["recommendations"]))

for r in response["recommendations"]:
    print(r["name"])

1
Graduate Scenarios


In [32]:
print(len(agent.state.previous_results))

1
